# ETH Long-Term Value & Price Research

Long-horizon Ether analysis: **on-chain activity**, **macro / monetary regimes**, an ETH-adapted
**stock-to-flow** framing (net issuance post-Merge), a protocol **fee / burn DCF**, early-signal
detection for multi-quarter moves, and calibrated **1y / 2y / 4y** forecasts.

**Thesis.** Post-Merge ETH is a hybrid monetary + cash-flow asset. Net issuance (staking mint − EIP-1559
burn) sets the supply schedule; L1 fees / burn are a crude cash-flow proxy; global liquidity and rates
still dominate multi-quarter risk appetite. Early signals come from *leading* macro momentum and
on-chain activity — not from coincident price transforms.

**Pipeline**
1. Load ETH OHLCV (DuckDB) → weekly frame; optional open interest from `mart_signals_daily`.
2. FRED monetary indicators → Global Liquidity Index + regime / early-warning (Macro-style, ETH-targeted).
3. Notebook-local ETH on-chain fetch + cache (supply, burn, staking, fees, TVL) with synthetic fallbacks.
4. **Stock-to-flow** from circulating supply / annualized net issuance (deflationary-regime handling).
5. **DCF** fair-value band from annualized fee revenue discounted off the Treasury curve.
6. Lead/lag + Granger + conditional forward-return backtests + Probit for favorable multi-quarter regimes.
7. HAC OLS + ARIMA + conformal bootstrap fans at 1y / 2y / 4y.

> **Run top-to-bottom.** Needs `uv sync --extra notebook --extra forecast` and a populated DuckDB
> (`ccquant sync backfill --interval 1d --force` if history is short). Set `FRED_API_KEY` for real macro; optional
> `ETHERSCAN_API_KEY` / `GLASSNODE_API_KEY`. Without keys the notebook degrades to synthetic series
> so the pipeline still runs — **set keys for research-grade numbers**.

**Caveat:** Do **not** use BTC on-chain columns from `mart_signals_daily` for ETH — those are Bitcoin
metrics joined by date only.


In [1]:
from __future__ import annotations

import json
import os
import time
import warnings
from datetime import UTC, date, datetime, timedelta
from pathlib import Path
from typing import Any

import httpx
import numpy as np
import plotly.graph_objects as go
import polars as pl
import statsmodels.api as sm
from dotenv import load_dotenv
from plotly.subplots import make_subplots
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import grangercausalitytests

from ccquant.forecasting import load_daily_panel, load_signals_panel

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- Config ----------------------------------------------------------------
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent

load_dotenv(_root / ".env", override=True)  # strip #comments; win over VS Code env

DB_PATH = os.environ.get("CCQUANT_DB", str(_root / "data" / "ccquant.duckdb"))
FRED_API_KEY = os.environ.get("FRED_API_KEY", "").strip()
ETHERSCAN_API_KEY = os.environ.get("ETHERSCAN_API_KEY", "").strip()
GLASSNODE_API_KEY = os.environ.get("GLASSNODE_API_KEY", "").strip()
CG_DEMO_API_KEY = os.environ.get("CG_DEMO_API_KEY", "").strip()
# Required when displaying CoinGecko market data (Demo/API):
# https://brand.coingecko.com/resources/attribution-guide
COINGECKO_ATTRIBUTION = (
    "Data provided by CoinGecko (https://www.coingecko.com)"
)

CACHE_DIR = _root / "data" / "eth_onchain_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_MAX_AGE_H = 12.0

WEEKLY_FREQ = "1w"
LIQ_LOOKBACK = 52
MOM_LOOKBACK = 12
LEAD_MIN, LEAD_MAX = -26, 26
FWD_HORIZONS_WK = [4, 13, 26, 52]
PROBIT_HORIZON_WK = 13

HORIZONS = [365, 730, 1461]  # 1y, 2y, ~4y
BOOTSTRAP_DRAWS = 500
CALIB_BOOTSTRAP_DRAWS = 200
CALIB_STEP_DAYS = 90
RNG_SEED = 42

MERGE_DATE = date(2022, 9, 15)
ETH_COLOR = "#627EEA"
ETH_RISK_PREMIUM = 0.08  # default equity-like premium over DGS10 for DCF

print(f"DB: {DB_PATH}")
print(f"FRED_API_KEY set: {bool(FRED_API_KEY)}")
print(f"ETHERSCAN key:    {'set' if ETHERSCAN_API_KEY else 'NOT set'}")
print(f"GLASSNODE key:    {'set' if GLASSNODE_API_KEY else 'NOT set (optional)'}")
print(f"CG_DEMO key:      {'set' if CG_DEMO_API_KEY else 'NOT set (CoinGecko mcap needs free Demo key)'}")
print(f"On-chain cache:   {CACHE_DIR}")


DB: /home/ricka/Git/GitHub/ccquant/data/ccquant.duckdb
FRED_API_KEY set: True
ETHERSCAN key:    set
GLASSNODE key:    NOT set (optional)
CG_DEMO key:      set
On-chain cache:   /home/ricka/Git/GitHub/ccquant/data/eth_onchain_cache


## 1. Load ETH market panel

Daily OHLCV from local DuckDB, collapsed to weekly (last close, summed volume). Weekly matches
macro lead/lag resolution and the Fed balance-sheet frequency.


In [2]:
panel = load_daily_panel(DB_PATH)
print(f"Full panel: {panel.shape}  symbols: {panel['symbol'].n_unique()}")

eth = (
    panel.filter(pl.col("symbol") == "ETH")
    .sort("date")
    .unique(subset=["date"])
    .with_columns(np.log(pl.col("close")).alias("log_close"))
)
if eth.height == 0:
    raise RuntimeError(
        "No ETH rows in DuckDB. Run: uv run ccquant sync universe && "
        "uv run ccquant sync backfill --interval 1d --force"
    )
print(f"ETH daily: {eth.height} rows  {eth['date'].min()} -> {eth['date'].max()}")

eth_weekly = (
    eth.with_columns(pl.col("date").dt.truncate(WEEKLY_FREQ).alias("week"))
    .group_by("week")
    .agg(
        pl.col("close").last().alias("close"),
        pl.col("volume").sum().alias("volume"),
    )
    .sort("week")
    .with_columns(
        np.log(pl.col("close")).alias("log_close"),
        (pl.col("close") / pl.col("close").shift(1) - 1.0).alias("wk_return"),
    )
)
eth_weekly = eth_weekly.with_columns(
    (pl.col("log_close") - pl.col("log_close").cum_max()).alias("log_drawdown")
)
print(f"ETH weekly: {eth_weekly.height} rows  {eth_weekly['week'].min()} -> {eth_weekly['week'].max()}")
eth_weekly.tail(3)

# Adaptive lookbacks when local ETH history is short (e.g. fresh DB / CI fixture)
_n_wk = eth_weekly.height
LIQ_LOOKBACK = min(LIQ_LOOKBACK, max(4, _n_wk // 3))
MOM_LOOKBACK = min(MOM_LOOKBACK, max(2, _n_wk // 5))
if _n_wk < 52:
    print(f"WARNING: only {_n_wk} ETH weeks in DB — using LIQ_LOOKBACK={LIQ_LOOKBACK}, "
          f"MOM_LOOKBACK={MOM_LOOKBACK}.")
    print("  Fix: uv run ccquant sync backfill --interval 1d --force --top 50")
    print("  (Binance HTTP 451 is normal in some regions; Coinbase/CoinGecko fallback is used.)")


Full panel: (90, 8)  symbols: 3
ETH daily: 30 rows  2026-06-12 -> 2026-07-11
ETH weekly: 5 rows  2026-06-08 -> 2026-07-06
  Fix: uv run ccquant sync backfill --interval 1d --force --top 50
  (Binance HTTP 451 is normal in some regions; Coinbase/CoinGecko fallback is used.)


### Optional: ETH open interest

Loaded from `main_marts.mart_signals_daily` when dbt has been built. Skipped gracefully otherwise.
BTC on-chain columns in that mart are **ignored**.


In [3]:
oi_weekly: pl.DataFrame | None = None
try:
    signals = load_signals_panel(DB_PATH)
    eth_sig = signals.filter(pl.col("symbol") == "ETH").sort("date")
    if (
        "total_open_interest_usd" in eth_sig.columns
        and eth_sig["total_open_interest_usd"].drop_nulls().len() > 0
    ):
        oi_weekly = (
            eth_sig.with_columns(pl.col("date").dt.truncate(WEEKLY_FREQ).alias("week"))
            .group_by("week")
            .agg(pl.col("total_open_interest_usd").last().alias("oi_usd"))
            .sort("week")
        )
        print(f"ETH OI weekly: {oi_weekly.height} rows")
    else:
        print("mart_signals_daily present but no ETH OI — skipping OI block.")
except Exception as exc:
    print(f"load_signals_panel unavailable ({exc}) — skipping OI (run dbt build if needed).")


ETH OI weekly: 5 rows


## 2. FRED monetary indicators

Same eight series as `Macro.ipynb` / `config.FRED_SERIES`. Synthetic fallback if `FRED_API_KEY` is unset.

| Series | What | Freq |
|---|---|---|
| `M2SL` | M2 money stock | monthly |
| `WALCL` | Fed total assets | weekly |
| `DGS10` / `DGS2` | 10Y / 2Y Treasury yield | daily |
| `T10YIE` | 10Y breakeven inflation | daily |
| `FEDFUNDS` | effective Fed funds | monthly |
| `DTWEXBGS` | trade-weighted USD | daily |
| `VIXCLS` | VIX | daily |


In [4]:
FRED_SERIES: dict[str, dict[str, Any]] = {
    "M2SL":     {"label": "M2 money stock", "freq": "monthly"},
    "WALCL":    {"label": "Fed total assets", "freq": "weekly"},
    "DGS10":    {"label": "10Y Treasury yield", "freq": "daily"},
    "DGS2":     {"label": "2Y Treasury yield", "freq": "daily"},
    "T10YIE":   {"label": "10Y breakeven inflation", "freq": "daily"},
    "FEDFUNDS": {"label": "Effective Fed funds rate", "freq": "monthly"},
    "DTWEXBGS": {"label": "Trade-weighted USD (broad)", "freq": "daily"},
    "VIXCLS":   {"label": "VIX", "freq": "daily"},
}

def fetch_fred_series(series_id: str, api_key: str, start: str) -> pl.DataFrame:
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id, "api_key": api_key, "file_type": "json",
        "observation_start": start,
    }
    with httpx.Client(timeout=30.0) as client:
        resp = client.get(url, params=params)
        resp.raise_for_status()
        obs = resp.json()["observations"]
    rows = [
        {"date": date.fromisoformat(o["date"]), series_id: float(o["value"])}
        for o in obs if o["value"] not in {".", ""}
    ]
    return pl.DataFrame(rows).sort("date")

fred_frames: dict[str, pl.DataFrame | None] = {}
if FRED_API_KEY:
    start_date = min(eth["date"].min() - timedelta(days=60), date(2015, 1, 1)).isoformat()
    print("Fetching FRED series...")
    for sid in FRED_SERIES:
        try:
            df = fetch_fred_series(sid, FRED_API_KEY, start_date)
            fred_frames[sid] = df
            print(f"  {sid:>10} ({FRED_SERIES[sid]['freq']:>7}): {df.height} obs")
        except Exception as exc:
            print(f"  {sid:>10}: FAILED ({exc})")
            fred_frames[sid] = None
else:
    print("FRED_API_KEY not set — using synthetic macro series.")
    fred_frames = {sid: None for sid in FRED_SERIES}


Fetching FRED series...
        M2SL (monthly): 137 obs
       WALCL ( weekly): 601 obs
       DGS10 (  daily): 2880 obs
        DGS2 (  daily): 2880 obs
      T10YIE (  daily): 2881 obs
    FEDFUNDS (monthly): 138 obs
    DTWEXBGS (  daily): 2867 obs
      VIXCLS (  daily): 2927 obs


In [5]:
def synthetic_macro(dates: pl.Series, sid: str) -> pl.DataFrame:
    rng = np.random.default_rng(abs(hash(sid)) % (2**32))
    n = dates.len()
    t = np.arange(n)
    base = {
        "M2SL": 13.0e12, "WALCL": 7.5e12, "DGS10": 2.5, "DGS2": 2.5,
        "T10YIE": 2.2, "FEDFUNDS": 2.0, "DTWEXBGS": 110.0, "VIXCLS": 18.0,
    }[sid]
    drift = {"M2SL": 0.06 / 52, "WALCL": 0.04 / 52}.get(sid, 0.0)
    cyc = 0.10 * np.sin(2 * np.pi * t / 260.0)
    noise = rng.normal(0, 0.02, n)
    if sid in {"M2SL", "WALCL"}:
        val = base * np.exp((drift + cyc * 0.05 + noise * 0.01) * t)
    elif sid in {"DGS10", "DGS2", "T10YIE", "FEDFUNDS"}:
        val = np.clip(base + 1.5 * cyc + np.cumsum(noise) * 0.4, 0.05, None)
    elif sid == "VIXCLS":
        val = np.clip(base * np.exp(0.3 * cyc + np.cumsum(noise) * 0.5), 9.0, None)
    else:
        val = base + 5.0 * cyc + np.cumsum(noise) * 0.5
    return pl.DataFrame({"date": dates.to_list(), sid: val}).sort("date")

week_spine = eth_weekly.select(pl.col("week").alias("date")).sort("date")
macro_weekly = week_spine
for sid in FRED_SERIES:
    df = fred_frames[sid]
    if df is None:
        df = synthetic_macro(week_spine["date"], sid)
    macro_weekly = macro_weekly.join_asof(df.sort("date"), on="date", strategy="backward")
macro_weekly = macro_weekly.with_columns(*[pl.col(sid).forward_fill() for sid in FRED_SERIES])
print(f"Macro weekly frame: {macro_weekly.shape}")
macro_weekly.tail(3)


Macro weekly frame: (5, 9)


date,M2SL,WALCL,DGS10,DGS2,T10YIE,FEDFUNDS,DTWEXBGS,VIXCLS
date,f64,f64,f64,f64,f64,f64,f64,f64
2026-06-22,23052.3,6.736424e6,4.51,4.24,2.23,3.63,120.5463,17.28
2026-06-29,23052.3,6.735645e6,4.38,4.1,2.22,3.63,120.9525,17.65
2026-07-06,23052.3,6.724564e6,4.48,4.13,2.24,3.63,120.6902,15.57


## 3. Macro regime features + Global Liquidity Index

**L = z(M2 growth) + z(Fed BS growth) − z(Δreal 10y)** — retargeted to ETH weekly returns.


In [6]:
def z_expr(col: str) -> pl.Expr:
    return (pl.col(col) - pl.col(col).mean()) / (pl.col(col).std() + 1e-12)

macro = macro_weekly.with_columns(
    (pl.col("DGS10") - pl.col("T10YIE")).alias("real_10y"),
    (pl.col("DGS10") - pl.col("DGS2")).alias("curve_10y_2y"),
).with_columns(
    (np.log(pl.col("M2SL")) - np.log(pl.col("M2SL")).shift(LIQ_LOOKBACK)).alias("m2_grow_yoy"),
    (np.log(pl.col("WALCL")) - np.log(pl.col("WALCL")).shift(LIQ_LOOKBACK)).alias("fedbs_grow_yoy"),
    (pl.col("real_10y") - pl.col("real_10y").shift(LIQ_LOOKBACK)).alias("real_rate_delta"),
    (pl.col("real_10y") - pl.col("real_10y").shift(MOM_LOOKBACK)).alias("real_10y_mom"),
    (pl.col("curve_10y_2y") - pl.col("curve_10y_2y").shift(MOM_LOOKBACK)).alias("curve_mom"),
    (np.log(pl.col("DTWEXBGS")) - np.log(pl.col("DTWEXBGS")).shift(MOM_LOOKBACK)).alias("dxy_mom"),
    (pl.col("VIXCLS") - pl.col("VIXCLS").shift(MOM_LOOKBACK)).alias("vix_mom"),
).with_columns(
    (pl.col("m2_grow_yoy") - pl.col("m2_grow_yoy").shift(MOM_LOOKBACK)).alias("m2_grow_mom"),
)

macro = macro.join(
    eth_weekly.select("week", "close", "log_close", "wk_return", "log_drawdown", "volume"),
    left_on="date", right_on="week", how="left",
)
if oi_weekly is not None:
    macro = macro.join(oi_weekly, left_on="date", right_on="week", how="left")
# Prefer growth features when available; fall back to levels if lookback exceeds history
_need = ["close", "real_10y"]
if macro["m2_grow_yoy"].drop_nulls().len() >= 4:
    _need.append("m2_grow_yoy")
macro = macro.drop_nulls(subset=_need)
if macro.height < 4:
    raise RuntimeError(
        f"Insufficient ETH/macro overlap ({macro.height} rows). "
        "Backfill more daily history: uv run ccquant sync backfill --interval 1d --force"
    )

if "m2_grow_yoy" in _need:
    macro = macro.with_columns(
        (z_expr("m2_grow_yoy") + z_expr("fedbs_grow_yoy") - z_expr("real_rate_delta")).alias("liq_raw")
    )
else:
    # Ultra-short history: level-based liquidity proxy
    macro = macro.with_columns(
        (z_expr("M2SL") + z_expr("WALCL") - z_expr("real_10y")).alias("liq_raw")
    )

_liq = macro["liq_raw"].drop_nulls()
mu, sd = float(_liq.mean()), float(_liq.std() or 1.0)
macro = macro.with_columns(
    ((pl.col("liq_raw") - mu) / (sd if sd > 1e-12 else 1.0)).alias("liq_index"),
).with_columns(
    (pl.col("liq_index") - pl.col("liq_index").shift(MOM_LOOKBACK)).alias("liq_mom"),
).with_columns(
    ((pl.col("liq_mom") > 0).cast(pl.Int8)).alias("regime"),
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=macro["date"], y=macro["liq_index"], name="Liquidity index",
                         line=dict(color="#6ea8ff", width=1.8), yaxis="y"))
fig.add_trace(go.Scatter(x=macro["date"], y=macro["close"], name="ETH",
                         line=dict(color=ETH_COLOR, width=1.5), yaxis="y2"))
fig.update_layout(
    title="Global Liquidity Composite vs ETH (weekly)",
    template="plotly_dark", height=450,
    yaxis=dict(title="liquidity z"),
    yaxis2=dict(title="ETH $", overlaying="y", side="right", type="log"),
    legend=dict(orientation="h", y=1.08),
)
fig.show()
print(f"corr(liq_index, log ETH) = {float(macro.select(pl.corr('liq_index', 'log_close')).item()):.3f}")
print(f"Merged macro+ETH: {macro.shape}  {macro['date'].min()} -> {macro['date'].max()}")


corr(liq_index, log ETH) = nan
Merged macro+ETH: (5, 29)  2026-06-08 -> 2026-07-06


## 4. Ethereum on-chain indicators (notebook-local)

Fetch + cache under `data/eth_onchain_cache/` (gitignored via `data/`). Prefer free/public APIs;
degrade to synthetic post-Merge paths when sources fail.

| Metric | Role | Source strategy |
|---|---|---|
| Circulating supply | S2F stock | CoinGecko Demo (`CG_DEMO_API_KEY`) market chart / synthetic |
| Net issuance (issuance − burn) | S2F flow | Ultrasound-style / synthetic activity-scaled |
| Staked share | Supply lock | Synthetic or Glassnode if keyed |
| L1 fee revenue USD | DCF cash-flow | Volume × fee intensity / Etherscan fallback |
| Active addresses / tx proxy | Activity | Synthetic scaled to volume |
| DeFi TVL | Demand | DefiLlama `historicalChainTvl/Ethereum` |

> Exploratory research layer — not a production ingestion pipeline. A future
> `sync eth-onchain` + `chain` column on `onchain_series` can promote this.

> When CoinGecko market data is used (Demo key): **Data provided by [CoinGecko](https://www.coingecko.com)**. See their [attribution guide](https://brand.coingecko.com/resources/attribution-guide).


In [7]:
def _cache_path(name: str) -> Path:
    return CACHE_DIR / f"{name}.json"

def _cache_fresh(path: Path) -> bool:
    if not path.exists():
        return False
    age_h = (time.time() - path.stat().st_mtime) / 3600.0
    return age_h < CACHE_MAX_AGE_H

def _read_cache(name: str) -> list[dict[str, Any]] | None:
    path = _cache_path(name)
    if not _cache_fresh(path):
        return None
    return json.loads(path.read_text())

def _write_cache(name: str, rows: list[dict[str, Any]]) -> None:
    _cache_path(name).write_text(json.dumps(rows))

def rows_to_df(rows: list[dict[str, Any]], value_col: str) -> pl.DataFrame:
    return pl.DataFrame({
        "date": [date.fromisoformat(r["date"]) for r in rows],
        value_col: [float(r["value"]) for r in rows],
    }).sort("date")

def fetch_defillama_tvl() -> pl.DataFrame | None:
    cached = _read_cache("defillama_tvl")
    if cached is not None:
        print("  TVL: cache hit")
        return rows_to_df(cached, "tvl_usd")
    try:
        with httpx.Client(timeout=45.0) as client:
            resp = client.get("https://api.llama.fi/v2/historicalChainTvl/Ethereum")
            resp.raise_for_status()
            data = resp.json()
        rows = [
            {
                "date": datetime.fromtimestamp(int(p["date"]), tz=UTC).date().isoformat(),
                "value": float(p["tvl"]),
            }
            for p in data if "date" in p and "tvl" in p
        ]
        if not rows:
            return None
        _write_cache("defillama_tvl", rows)
        print(f"  TVL: DefiLlama {len(rows)} pts")
        return rows_to_df(rows, "tvl_usd")
    except Exception as exc:
        print(f"  TVL: FAILED ({exc})")
        return None

def fetch_coingecko_mcap() -> pl.DataFrame | None:
    """Market-cap series; supply ≈ mcap / price when joined later.

    CoinGecko public calls often return HTTP 401 without a free Demo API key.
    Set ``CG_DEMO_API_KEY`` in ``.env`` (https://www.coingecko.com/en/api/pricing).
    """
    cached = _read_cache("coingecko_mcap")
    if cached is not None:
        print("  mcap: cache hit")
        return rows_to_df(cached, "mcap_usd")
    if not CG_DEMO_API_KEY:
        print(
            "  mcap: skipped — set CG_DEMO_API_KEY in .env for CoinGecko "
            "(free Demo key). Using synthetic supply instead."
        )
        return None
    headers = {"x-cg-demo-api-key": CG_DEMO_API_KEY}
    try:
        with httpx.Client(timeout=45.0, headers=headers) as client:
            resp = client.get(
                "https://api.coingecko.com/api/v3/coins/ethereum/market_chart",
                params={"vs_currency": "usd", "days": "max", "interval": "daily"},
            )
            if resp.status_code == 401:
                print(
                    "  mcap: CoinGecko 401 — check CG_DEMO_API_KEY "
                    "(Demo plan header x-cg-demo-api-key). Using synthetic supply."
                )
                return None
            resp.raise_for_status()
            data = resp.json()
        rows = [
            {
                "date": datetime.fromtimestamp(ts / 1000.0, tz=UTC).date().isoformat(),
                "value": float(val),
            }
            for ts, val in data.get("market_caps", [])
        ]
        # dedupe by date (keep last)
        by_date: dict[str, float] = {}
        for r in rows:
            by_date[r["date"]] = r["value"]
        rows = [{"date": d, "value": v} for d, v in sorted(by_date.items())]
        if not rows:
            return None
        _write_cache("coingecko_mcap", rows)
        print(f"  mcap: CoinGecko {len(rows)} pts")
        print(f"  {COINGECKO_ATTRIBUTION}")
        return rows_to_df(rows, "mcap_usd")
    except Exception as exc:
        print(f"  mcap: FAILED ({exc}) — using synthetic supply")
        return None

def fetch_glassnode_metric(path: str, metric: str) -> pl.DataFrame | None:
    if not GLASSNODE_API_KEY:
        return None
    cache_name = f"glassnode_{metric}"
    cached = _read_cache(cache_name)
    if cached is not None:
        print(f"  glassnode {metric}: cache hit")
        return rows_to_df(cached, metric)
    try:
        with httpx.Client(timeout=45.0) as client:
            resp = client.get(
                f"https://api.glassnode.com/v1/metrics/{path}",
                params={"a": "ETH", "api_key": GLASSNODE_API_KEY, "i": "24h"},
            )
            resp.raise_for_status()
            data = resp.json()
        rows = [
            {
                "date": datetime.fromtimestamp(int(p["t"]), tz=UTC).date().isoformat(),
                "value": float(p["v"]),
            }
            for p in data if p.get("v") is not None
        ]
        _write_cache(cache_name, rows)
        print(f"  glassnode {metric}: {len(rows)} pts")
        time.sleep(6.0)
        return rows_to_df(rows, metric)
    except Exception as exc:
        print(f"  glassnode {metric}: FAILED ({exc})")
        return None

print("Fetching ETH on-chain series...")
tvl_df = fetch_defillama_tvl()
mcap_df = fetch_coingecko_mcap()
gn_supply = fetch_glassnode_metric("supply/current", "supply_gn")
gn_burn = fetch_glassnode_metric("fees/gas_used_sum", "gas_used")  # activity proxy if available


Fetching ETH on-chain series...
  TVL: cache hit
  mcap: CoinGecko 401 — check CG_DEMO_API_KEY (Demo plan header x-cg-demo-api-key). Using synthetic supply.


In [8]:
def synthetic_eth_onchain(dates: pl.Series, close: pl.Series, volume: pl.Series) -> pl.DataFrame:
    """Plausible post-Merge ETH fundamentals so the notebook runs offline."""
    rng = np.random.default_rng(RNG_SEED + 17)
    n = dates.len()
    t = np.arange(n, dtype=float)
    close_np = close.to_numpy().astype(float)
    vol_np = volume.to_numpy().astype(float)
    vol_z = (vol_np - np.nanmean(vol_np)) / (np.nanstd(vol_np) + 1e-12)
    vol_z = np.nan_to_num(vol_z, nan=0.0)

    # Circulating supply ~120M at Merge, slow drift from net issuance
    supply = np.full(n, 120.0e6)
    # Pre-merge: higher issuance (~13k ETH/day ≈ 4.75M/yr); post-merge ~0.5%/yr staking mint
    issuance_daily = np.where(
        np.array([d >= MERGE_DATE for d in dates.to_list()]),
        1700.0,  # ~post-merge staking issuance
        13000.0,
    )
    # Burn scales with activity (volume z) and a fee-intensity cycle
    fee_intensity = 0.35 + 0.25 * (0.5 + 0.5 * np.sin(2 * np.pi * t / 78.0))  # ~1.5y cycle
    burn_daily = np.clip(issuance_daily * (0.4 + 0.5 * np.tanh(vol_z)) * fee_intensity, 0, None)
    burn_daily = burn_daily * (1.0 + 0.05 * rng.normal(size=n))
    net_daily = issuance_daily - burn_daily
    for i in range(1, n):
        supply[i] = supply[i - 1] + net_daily[i] * 7.0  # weekly steps

    staked_share = np.clip(0.12 + 0.18 * (t / max(n - 1, 1)) + 0.02 * rng.normal(size=n), 0.05, 0.40)
    # Fee revenue USD ≈ volume * intensity * price scale factor
    # ~$1–5B annual fee run-rate scaled by activity (weekly CF)
    fee_ann = 2.5e9 * fee_intensity * (1.0 + 0.35 * np.tanh(vol_z))
    fee_usd = np.clip(fee_ann / 52.0, 1e6, None)
    active_addr = np.clip(400_000 * np.exp(0.15 * vol_z) * (1 + 0.1 * np.sin(2 * np.pi * t / 52)), 50_000, None)
    tx_count = active_addr * (1.8 + 0.3 * vol_z)
    tvl = np.clip(close_np * 8.0e6 * (1.0 + 0.3 * vol_z), 1e9, None)

    return pl.DataFrame({
        "date": dates.to_list(),
        "supply": supply,
        "issuance_eth_wk": issuance_daily * 7.0,
        "burn_eth_wk": burn_daily * 7.0,
        "net_issuance_eth_wk": net_daily * 7.0,
        "staked_share": staked_share,
        "fee_revenue_usd_wk": fee_usd,
        "active_addresses": active_addr,
        "tx_count": tx_count,
        "tvl_usd": tvl,
    }).sort("date")

# Build on-chain weekly frame aligned to macro date spine
oc = synthetic_eth_onchain(macro["date"], macro["close"], macro["volume"])
oc_source = "synthetic"

# Overlay real TVL if available
if tvl_df is not None:
    oc = oc.drop("tvl_usd").join_asof(tvl_df.sort("date"), on="date", strategy="backward")
    oc = oc.with_columns(pl.col("tvl_usd").forward_fill().backward_fill())
    oc_source = "synthetic+defillama"

# Overlay supply from mcap/price or Glassnode
if gn_supply is not None:
    oc = oc.drop("supply").join_asof(
        gn_supply.rename({"supply_gn": "supply"}).sort("date"), on="date", strategy="backward"
    )
    oc = oc.with_columns(pl.col("supply").forward_fill())
    oc_source += "+glassnode_supply"
elif mcap_df is not None:
    mcap_join = macro.select("date", "close").join_asof(mcap_df.sort("date"), on="date", strategy="backward")
    supply_est = mcap_join.with_columns(
        (pl.col("mcap_usd") / pl.col("close")).alias("supply_est")
    ).select("date", "supply_est")
    oc = oc.join(supply_est, on="date", how="left").with_columns(
        pl.when(pl.col("supply_est").is_not_null() & (pl.col("supply_est") > 50e6))
        .then(pl.col("supply_est"))
        .otherwise(pl.col("supply"))
        .alias("supply")
    ).drop("supply_est")
    oc_source += "+coingecko_mcap"
    print(COINGECKO_ATTRIBUTION)

print(f"On-chain frame: {oc.shape}  source={oc_source}")
oc.select("date", "supply", "net_issuance_eth_wk", "fee_revenue_usd_wk", "tvl_usd", "staked_share").tail(3)


On-chain frame: (5, 10)  source=synthetic+defillama


date,supply,net_issuance_eth_wk,fee_revenue_usd_wk,tvl_usd,staked_share
date,f64,f64,f64,f64,f64
2026-06-22,1.2002e8,7481.261811,2.8537e7,3.8276e10,0.188214
2026-06-29,1.2002e8,7791.637649,2.9106e7,3.7237e10,0.298305
2026-07-06,1.2003e8,9565.296045,2.4740e7,4.0001e10,0.305046


In [9]:
# Merge on-chain into working frame `df`
df = macro.join(oc, on="date", how="left").sort("date")

df = df.with_columns(
    (np.log(pl.col("tvl_usd") + 1) - np.log(pl.col("tvl_usd") + 1).shift(MOM_LOOKBACK)).alias("tvl_mom"),
    (np.log(pl.col("fee_revenue_usd_wk") + 1) - np.log(pl.col("fee_revenue_usd_wk") + 1).shift(MOM_LOOKBACK)).alias("fee_mom"),
    (np.log(pl.col("active_addresses") + 1) - np.log(pl.col("active_addresses") + 1).shift(MOM_LOOKBACK)).alias("addr_mom"),
    (pl.col("staked_share") - pl.col("staked_share").shift(MOM_LOOKBACK)).alias("stake_mom"),
)

# ETH cycle-activity composite (NOT BTC hashrate/MVRV clones)
df = df.with_columns(
    (
        z_expr("fee_mom") + z_expr("tvl_mom") + z_expr("addr_mom") + z_expr("stake_mom")
    ).alias("activity_raw")
)
_ar = df["activity_raw"].drop_nulls()
mu_a = float(_ar.mean()) if _ar.len() else 0.0
sd_a = float(_ar.std() or 1.0) if _ar.len() else 1.0
df = df.with_columns(
    ((pl.col("activity_raw") - mu_a) / (sd_a if sd_a > 1e-12 else 1.0)).alias("activity_composite")
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["date"], y=df["activity_composite"], name="ETH activity composite",
                         line=dict(color="#39ff14", width=1.6), yaxis="y"))
fig.add_trace(go.Scatter(x=df["date"], y=df["close"], name="ETH",
                         line=dict(color=ETH_COLOR, width=1.5), yaxis="y2"))
fig.update_layout(
    title="ETH cycle-activity composite vs price (fees + TVL + addresses + staking)",
    template="plotly_dark", height=450,
    yaxis=dict(title="activity z"),
    yaxis2=dict(title="ETH $", overlaying="y", side="right", type="log"),
    legend=dict(orientation="h", y=1.08),
)
fig.show()
print(f"Working frame: {df.shape}")


Working frame: (5, 44)


## 5. Ethereum stock-to-flow framing

Post-Merge ETH is **not** a fixed-halving asset. Frame S2F as:

$$\mathrm{S2F} = \frac{\text{circulating supply}}{\max(\text{annualized net issuance},\,\varepsilon)}$$

where **net issuance = issuance − burn**. When net issuance ≤ 0 (ultrasound / net-deflationary),
flag a deflationary regime and cap S2F at a high sentinel rather than exploding the ratio.

Treat this as **exploratory framing**, not PlanB dogma.


In [10]:
S2F_EPS = 1e3  # ETH/year floor for ratio stability
S2F_CAP = 500.0  # sentinel for near-zero / negative net issuance

df = df.with_columns(
    (pl.col("net_issuance_eth_wk") * 52.0).alias("net_issuance_ann"),
    (pl.col("issuance_eth_wk") * 52.0).alias("issuance_ann"),
    (pl.col("burn_eth_wk") * 52.0).alias("burn_ann"),
).with_columns(
    (pl.col("net_issuance_ann") <= 0).cast(pl.Int8).alias("is_deflationary"),
).with_columns(
    pl.when(pl.col("net_issuance_ann") > S2F_EPS)
    .then(pl.col("supply") / pl.col("net_issuance_ann"))
    .otherwise(pl.lit(S2F_CAP))
    .alias("s2f"),
).with_columns(
    np.log(pl.col("s2f").clip(1e-6, None)).alias("log_s2f"),
    (pl.col("net_issuance_ann") / pl.col("supply")).alias("net_issuance_rate"),
)

# Exploratory HAC OLS: log(price) ~ log(S2F)
s2f_fit = df.select("log_close", "log_s2f", "is_deflationary").drop_nulls()
y_s2f = s2f_fit["log_close"].to_numpy()
X_s2f = sm.add_constant(s2f_fit["log_s2f"].to_numpy())
s2f_ols = sm.OLS(y_s2f, X_s2f).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
print("log(ETH) ~ const + log(S2F)  [HAC]")
print(s2f_ols.summary().tables[1])
print(f"R² = {s2f_ols.rsquared:.3f}  deflationary weeks = {int(df['is_deflationary'].sum())} / {df.height}")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("ETH price (log) + deflationary weeks", "S2F (capped) + net issuance rate"))
fig.add_trace(go.Scatter(x=df["date"], y=df["close"], name="ETH",
                         line=dict(color=ETH_COLOR, width=1.5)), row=1, col=1)
defl = df.filter(pl.col("is_deflationary") == 1)
if defl.height:
    fig.add_trace(go.Scatter(
        x=defl["date"], y=defl["close"], mode="markers", name="deflationary",
        marker=dict(color="#39ff14", size=5, symbol="diamond"),
    ), row=1, col=1)
fig.add_trace(go.Scatter(x=df["date"], y=df["s2f"], name="S2F",
                         line=dict(color="#f0c674", width=1.4)), row=2, col=1)
fig.add_trace(go.Scatter(x=df["date"], y=df["net_issuance_rate"] * 100, name="net issuance %/yr",
                         line=dict(color="#cc6666", width=1.2, dash="dot")), row=2, col=1)
fig.update_layout(template="plotly_dark", height=560, legend=dict(orientation="h", y=1.05),
                  title="ETH stock-to-flow framing (net issuance)")
fig.update_yaxes(type="log", row=1, col=1)
fig.update_yaxes(title_text="S2F / net issuance %", row=2, col=1)
fig.show()



log(ETH) ~ const + log(S2F)  [HAC]
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          8.0064   5.15e-14   1.55e+14      0.000       8.006       8.006
x1          1.413e-15   1.01e-14      0.140      0.888   -1.83e-14    2.11e-14
R² = -inf  deflationary weeks = 0 / 5


/home/ricka/Git/GitHub/ccquant/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


## 6. Discounted cash-flow valuation

Treat annualized L1 fee revenue as a cash-flow proxy (not GAAP equity earnings). Discount rate:

$$r = \mathrm{DGS10}/100 + \text{ETH risk premium}$$

with default premium = **8%** (`ETH_RISK_PREMIUM`). Gordon terminal:

$$V = \frac{\mathrm{CF}_{\mathrm{next}}}{r - g}, \quad \mathrm{CF}_{\mathrm{next}} = \mathrm{CF}_{\mathrm{ann}} \cdot (1+g)$$

Implied fair price = $V$ / circulating supply. Bear / base / bull growth scenarios; Plotly sensitivity heatmap.


In [11]:
G_SCENARIOS = {"bear": -0.02, "base": 0.03, "bull": 0.08}
PREMIUM_GRID = [0.04, 0.06, 0.08, 0.10, 0.12]
G_GRID = [-0.02, 0.00, 0.03, 0.05, 0.08]

df = df.with_columns(
    (pl.col("fee_revenue_usd_wk") * 52.0).alias("cf_ann_usd"),
    (pl.col("DGS10") / 100.0 + ETH_RISK_PREMIUM).alias("discount_r"),
)

for name, g in G_SCENARIOS.items():
    df = df.with_columns(
        pl.when(pl.col("discount_r") > g + 1e-4)
        .then(
            (pl.col("cf_ann_usd") * (1.0 + g))
            / (pl.col("discount_r") - g)
            / pl.col("supply")
        )
        .otherwise(None)
        .alias(f"dcf_price_{name}")
    )

df = df.with_columns(
    (pl.col("close") / pl.col("dcf_price_base") - 1.0).alias("dcf_gap"),
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["date"], y=df["close"], name="market",
                         line=dict(color=ETH_COLOR, width=2)))
fig.add_trace(go.Scatter(x=df["date"], y=df["dcf_price_bear"], name="DCF bear",
                         line=dict(color="#cc6666", width=1, dash="dot")))
fig.add_trace(go.Scatter(x=df["date"], y=df["dcf_price_base"], name="DCF base",
                         line=dict(color="#f0c674", width=1.5)))
fig.add_trace(go.Scatter(x=df["date"], y=df["dcf_price_bull"], name="DCF bull",
                         line=dict(color="#39ff14", width=1, dash="dot")))
# Fill band bear→bull
fig.add_trace(go.Scatter(
    x=df["date"].to_list() + df["date"].to_list()[::-1],
    y=df["dcf_price_bull"].to_list() + df["dcf_price_bear"].to_list()[::-1],
    fill="toself", fillcolor="rgba(98,126,234,0.12)", line=dict(color="rgba(0,0,0,0)"),
    name="DCF band", hoverinfo="skip",
))
fig.update_layout(
    title=f"ETH market price vs fee-DCF band (r = DGS10 + {ETH_RISK_PREMIUM:.0%} premium)",
    yaxis_type="log", template="plotly_dark", height=480,
    legend=dict(orientation="h", y=1.08),
)
fig.show()

# Sensitivity heatmap at latest observation
latest = df.drop_nulls(subset=["cf_ann_usd", "supply", "DGS10"]).tail(1)
cf0 = float(latest["cf_ann_usd"][0])
sup0 = float(latest["supply"][0])
dgs0 = float(latest["DGS10"][0]) / 100.0
z = []
for g in G_GRID:
    row = []
    for prem in PREMIUM_GRID:
        r = dgs0 + prem
        if r <= g + 1e-4:
            row.append(np.nan)
        else:
            row.append((cf0 * (1 + g)) / (r - g) / sup0)
    z.append(row)

fig_h = go.Figure(data=go.Heatmap(
    z=z, x=[f"{p:.0%}" for p in PREMIUM_GRID], y=[f"g={g:.0%}" for g in G_GRID],
    colorscale="Viridis", colorbar=dict(title="fair $"),
))
fig_h.update_layout(
    title=f"DCF implied price sensitivity (CF=${cf0/1e9:.2f}B, supply={sup0/1e6:.1f}M, DGS10={dgs0:.1%})",
    xaxis_title="ETH risk premium", template="plotly_dark", height=380,
)
fig_h.show()
_dcf_b = latest["dcf_price_base"][0]
_gap = latest["dcf_gap"][0]
if _dcf_b is None or (isinstance(_dcf_b, float) and not np.isfinite(_dcf_b)):
    print(f"Latest market ${float(latest['close'][0]):,.0f}  DCF base n/a (check CF / discount rate)")
else:
    print(f"Latest market ${float(latest['close'][0]):,.0f}  DCF base ${float(_dcf_b):,.0f}  "
          f"gap={float(_gap):+.1%}")


Latest market $3,000  DCF base $116  gap=+2476.3%


## 7. Early signals for long-term price action

Cross-correlate leading macro + on-chain + valuation signals against **forward 13-week ETH log returns**.
Build an early-warning composite, backtest conditional forward returns, and fit a Probit for
$P(\text{favorable multi-quarter regime})$.


In [12]:
df = df.with_columns(
    (pl.col("log_close").shift(-13) - pl.col("log_close")).alias("fwd_13w")
)

# Macro early-warn (same construction as Macro.ipynb)
df = df.with_columns(
    (-z_expr("real_10y_mom")).alias("_c1"),
    z_expr("curve_mom").alias("_c2"),
    z_expr("m2_grow_mom").alias("_c3"),
    (-z_expr("dxy_mom")).alias("_c4"),
).with_columns(
    (pl.col("_c1") + pl.col("_c2") + pl.col("_c3") + pl.col("_c4")).alias("early_warn_raw")
).drop(["_c1", "_c2", "_c3", "_c4"])
_ew = df["early_warn_raw"].drop_nulls()
mu_e = float(_ew.mean()) if _ew.len() else 0.0
sd_e = float(_ew.std() or 1.0) if _ew.len() else 1.0
df = df.with_columns(
    ((pl.col("early_warn_raw") - mu_e) / (sd_e if sd_e > 1e-12 else 1.0)).alias("early_warn")
)

# Combined ETH early signal: macro early-warn + activity momentum − DCF overvaluation
df = df.with_columns(
    (z_expr("early_warn") + z_expr("activity_composite") - z_expr("dcf_gap")).alias("eth_early_raw")
)
_ex = df["eth_early_raw"].drop_nulls()
mu_x = float(_ex.mean()) if _ex.len() else 0.0
sd_x = float(_ex.std() or 1.0) if _ex.len() else 1.0
df = df.with_columns(
    ((pl.col("eth_early_raw") - mu_x) / (sd_x if sd_x > 1e-12 else 1.0)).alias("eth_early")
)

lead_df = df.select(
    "date", "liq_index", "early_warn", "eth_early", "activity_composite",
    "net_issuance_rate", "log_s2f", "dcf_gap", "tvl_mom", "fee_mom",
    "fwd_13w", "wk_return",
).drop_nulls()

def cross_corr(x: np.ndarray, y: np.ndarray, lags: range) -> list[tuple[int, float]]:
    out = []
    n = len(y)
    for lag in lags:
        if lag < 0:
            a, b = x[-lag:], y[: n + lag]
        else:
            a, b = x[: n - lag], y[lag:]
        if len(a) > 20:
            out.append((lag, float(np.corrcoef(a, b)[0, 1])))
    return out

fwd13 = lead_df["fwd_13w"].to_numpy()
btc_ret = lead_df["wk_return"].to_numpy()  # name kept for parity; values are ETH
signals = {
    "liquidity index": lead_df["liq_index"].to_numpy(),
    "macro early-warn": lead_df["early_warn"].to_numpy(),
    "ETH early composite": lead_df["eth_early"].to_numpy(),
    "activity composite": lead_df["activity_composite"].to_numpy(),
    "net issuance rate": lead_df["net_issuance_rate"].to_numpy(),
    "log S2F": lead_df["log_s2f"].to_numpy(),
    "DCF gap": lead_df["dcf_gap"].to_numpy(),
    "TVL momentum": lead_df["tvl_mom"].to_numpy(),
    "fee momentum": lead_df["fee_mom"].to_numpy(),
}

lags = range(LEAD_MIN, LEAD_MAX + 1)
lead_results: dict[str, dict[str, Any]] = {}
print("Cross-correlation vs FORWARD 13-week ETH log return (lag>0 = signal leads):\n")
print(f"  {'signal':>22} {'lead lag':>8} {'lead corr':>10} {'@lag0':>8}")
if lead_df.height < 20:
    print(f"  (skipped — only {lead_df.height} weeks with forward returns; backfill more history)")
    for name in signals:
        lead_results[name] = {"lead_lag": 0, "lead_corr": 0.0, "cc": [(0, 0.0)]}
else:
    for name, sig in signals.items():
        cc = cross_corr(sig, fwd13, lags)
        lead_only = [(lg, c) for lg, c in cc if lg > 0]
        best_lead = max(lead_only, key=lambda t: abs(t[1])) if lead_only else (0, 0.0)
        lag0 = next((c for lg, c in cc if lg == 0), 0.0)
        lead_results[name] = {"lead_lag": best_lead[0], "lead_corr": best_lead[1], "cc": cc}
        print(f"  {name:>22} {best_lead[0]:>7}w {best_lead[1]:>+9.2f} {lag0:>+8.2f}")

    lag_axis = list(range(LEAD_MIN, LEAD_MAX + 1))
    mat, names = [], []
    for name, r in lead_results.items():
        cc_map = dict(r["cc"])
        mat.append([cc_map.get(lg, np.nan) for lg in lag_axis])
        names.append(name)
    fig = go.Figure(data=go.Heatmap(z=mat, x=lag_axis, y=names, colorscale="RdBu", zmid=0,
                                    colorbar=dict(title="corr")))
    fig.update_layout(
        title="Signal[t-lag] vs FORWARD 13w ETH return (lag>0 = leads)",
        xaxis_title="lag (weeks)", template="plotly_dark", height=460,
    )
    fig.show()


Cross-correlation vs FORWARD 13-week ETH log return (lag>0 = signal leads):

                  signal lead lag  lead corr    @lag0
  (skipped — only 0 weeks with forward returns; backfill more history)


In [13]:
print("Granger causality (signal -> ETH weekly return), maxlag=12:\n")
if lead_df.height < 30:
    print(f"  (skipped — only {lead_df.height} weeks; need ~30+ for Granger)")
else:
    for name in ["liquidity index", "ETH early composite", "activity composite", "fee momentum", "DCF gap"]:
        sig = signals[name]
        sig_diff = np.diff(sig, prepend=sig[0])
        try:
            res = grangercausalitytests(
                np.column_stack([btc_ret, sig_diff]), maxlag=min(12, lead_df.height // 3), verbose=False
            )
            maxlag = min(12, lead_df.height // 3)
            pvals = [res[lag][0]["ssr_ftest"][1] for lag in range(1, maxlag + 1)]
            best_lag = int(np.argmin(pvals) + 1)
            print(f"  {name:>22}: min p = {min(pvals):.3f} at lag {best_lag}w")
        except Exception as exc:
            print(f"  {name:>22}: failed ({exc})")

# Favorable multi-quarter regime: forward 13w return > 0 AND liquidity expanding
df = df.with_columns(
    (
        (pl.col("fwd_13w") > 0) & (pl.col("regime") == 1)
    ).cast(pl.Float64).alias("favorable_fwd")
)

for h in FWD_HORIZONS_WK:
    df = df.with_columns(
        (pl.col("log_close").shift(-h) - pl.col("log_close")).alias(f"fwd_{h}w")
    )

bt_rows: list[dict[str, Any]] = []
print("\nForward ETH log returns by liquidity regime:\n")
print(f"  {'horizon':>8} {'regime':>9} {'n':>5} {'mean':>8} {'median':>8} {'pos%':>6}")
for h in FWD_HORIZONS_WK:
    col = f"fwd_{h}w"
    for r, label in [(1, "expand"), (0, "contract")]:
        s = df.filter(pl.col("regime") == r).select(col).drop_nulls().to_series().to_numpy()
        if len(s) == 0:
            continue
        row = {
            "horizon": h, "regime": label, "n": len(s),
            "mean": float(np.mean(s)), "median": float(np.median(s)),
            "p25": float(np.percentile(s, 25)), "p75": float(np.percentile(s, 75)),
            "pos_pct": float((s > 0).mean()),
        }
        bt_rows.append(row)
        print(f"  {h:>7}w {label:>9} {len(s):>5} {row['mean']:+.3f} {row['median']:+.3f} "
              f"{row['pos_pct']:>5.0%}")
if not bt_rows:
    print("  (no complete forward-return windows yet — short history)")

df = df.with_columns(
    ((pl.col("eth_early") > 0) & (pl.col("eth_early").shift(1) <= 0)).alias("ew_bull_trigger")
)
trig = df.filter(pl.col("ew_bull_trigger"))
print(f"\nETH early-warn bullish triggers: {trig.height}")
print(f"  {'horizon':>8} {'n':>5} {'mean':>8} {'median':>8} {'pos%':>6}")
for h in FWD_HORIZONS_WK:
    s = trig.select(f"fwd_{h}w").drop_nulls().to_series().to_numpy()
    if len(s):
        print(f"  {h:>7}w {len(s):>5} {np.mean(s):+.3f} {np.median(s):+.3f} {(s > 0).mean():>5.0%}")


Granger causality (signal -> ETH weekly return), maxlag=12:

  (skipped — only 0 weeks; need ~30+ for Granger)

Forward ETH log returns by liquidity regime:

   horizon    regime     n     mean   median   pos%
  (no complete forward-return windows yet — short history)

ETH early-warn bullish triggers: 0
   horizon     n     mean   median   pos%


In [14]:
# Probit: P(favorable multi-quarter ETH regime in ~13w)
horizon = PROBIT_HORIZON_WK
feat_cols = [
    "real_10y", "curve_10y_2y", "m2_grow_yoy", "fedbs_grow_yoy",
    "early_warn", "activity_composite", "net_issuance_rate", "dcf_gap", "log_s2f",
]
fit_df = df.select(["favorable_fwd"] + feat_cols).drop_nulls()
y_p = fit_df["favorable_fwd"].to_numpy()
X_p = fit_df.select(feat_cols).to_numpy().astype(float)
mu_xp = X_p.mean(axis=0)
sd_xp = X_p.std(axis=0)
sd_xp[sd_xp < 1e-12] = 1.0
Xp_c = np.column_stack([np.ones(len(X_p)), (X_p - mu_xp) / sd_xp])

pos_rate = float(y_p.mean()) if len(y_p) else 0.0
oos_brier: list[float] = []
p_fav = float("nan")
probit = None
if len(y_p) < 30 or len(np.unique(y_p)) < 2:
    print(f"Probit skipped — need longer labeled history (n={len(y_p)}, "
          f"unique targets={len(np.unique(y_p)) if len(y_p) else 0}).")
else:
    probit = sm.Probit(y_p, Xp_c).fit(method="bfgs", maxiter=1000, disp=0)
    print("Probit coefficients (standardized features):")
    for nm, cf in zip(["const"] + feat_cols, probit.params, strict=True):
        print(f"  {nm:>20} {cf:+.3f}")
    print(f"\nPseudo R2 = {probit.prsquared:.3f}  n = {int(probit.nobs)}  base rate = {pos_rate:.0%}")
    min_train = min(80, max(20, len(Xp_c) // 2))
    for o in range(min_train, len(Xp_c) - horizon, 13):
        if len(np.unique(y_p[:o])) < 2:
            continue
        m = sm.Probit(y_p[:o], Xp_c[:o]).fit(method="bfgs", maxiter=500, disp=0)
        p = float(m.predict(Xp_c[o : o + 1])[0])
        oos_brier.append((p - y_p[o]) ** 2)
    naive_brier = pos_rate * (1 - pos_rate)
    if oos_brier:
        print(f"Walk-forward Brier: {np.mean(oos_brier):.3f} over {len(oos_brier)} origins "
              f"(naive = {naive_brier:.3f})")
    p_fav = float(probit.predict(Xp_c[-1:])[0])
    print(f"\nCurrent P(favorable ETH regime in {horizon}w): {p_fav:.0%}")


Probit skipped — need longer labeled history (n=0, unique targets=0).


/tmp/ipykernel_132060/102083712.py:10: RuntimeWarning: Mean of empty slice
  mu_xp = X_p.mean(axis=0)
/home/ricka/Git/GitHub/ccquant/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/home/ricka/Git/GitHub/ccquant/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/ricka/Git/GitHub/ccquant/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/ricka/Git/GitHub/ccquant/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:208: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


## 8. Long-horizon forecast (HAC OLS + ARIMA + bootstrap)

Daily modeling frame for cointegrating OLS on $\log(\text{ETH})$, ARIMA baseline, and
conformal-calibrated bootstrap fans at 1y / 2y / 4y — same structure as `BTC.ipynb`.


In [15]:
# Build daily modeling frame from ETH + weekly features asof-joined
daily_spine = eth.select("date", "close", "log_close").sort("date")

mw = macro_weekly.with_columns(
    np.log(pl.col("M2SL")).alias("log_m2"),
    np.log(pl.col("WALCL")).alias("log_fed_bs"),
    pl.col("DGS10").alias("yield_10y"),
).select("date", "log_m2", "log_fed_bs", "yield_10y")

oc_daily_feats = df.select(
    "date", "log_s2f", "net_issuance_rate", "is_deflationary",
    "activity_composite", "dcf_gap", "liq_index",
)

model_df = (
    daily_spine
    .join_asof(mw.sort("date"), on="date", strategy="backward")
    .join_asof(oc_daily_feats.sort("date"), on="date", strategy="backward")
    .sort("date")
    .with_columns(
        pl.col("log_m2").forward_fill(),
        pl.col("log_fed_bs").forward_fill(),
        pl.col("yield_10y").forward_fill(),
        pl.col("log_s2f").forward_fill(),
        pl.col("net_issuance_rate").forward_fill(),
        pl.col("activity_composite").forward_fill(),
        pl.col("dcf_gap").forward_fill(),
        pl.col("liq_index").forward_fill(),
        pl.col("is_deflationary").forward_fill().fill_null(0),
    )
    .drop_nulls(subset=["log_close", "log_m2", "log_s2f", "activity_composite"])
)
model_df = model_df.with_columns(
    ((pl.col("date") - pl.col("date").min()).dt.total_days()).alias("t")
)
print(f"Daily modeling frame: {model_df.shape}  {model_df['date'].min()} -> {model_df['date'].max()}")
model_df.describe()


Daily modeling frame: (20, 13)  2026-06-22 -> 2026-07-11


statistic,date,close,log_close,log_m2,log_fed_bs,yield_10y,log_s2f,net_issuance_rate,is_deflationary,activity_composite,dcf_gap,liq_index,t
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""20""",20.0,20.0,20.0,20.0,20.0,20.0,20.0,20.0,20.0,20.0,20.0,20.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""","""2026-07-01 12:00:00""",3000.0,8.006368,10.045521,15.722471,4.4555,5.643845,0.003559,0.0,0.01791,22.152404,0.187503,9.5
"""std""",null,0.0,0.0,0.0,0.000803,0.058172,0.107352,0.000397,0.0,0.854255,1.781648,0.864502,5.91608
"""min""","""2026-06-22""",3000.0,8.006368,10.045521,15.721278,4.38,5.486132,0.003241,0.0,-0.771562,20.665388,-0.598868,0.0
"""25%""","""2026-06-27""",3000.0,8.006368,10.045521,15.721278,4.38,5.486132,0.003241,0.0,-0.771562,20.665388,-0.598868,5.0
"""50%""","""2026-07-02""",3000.0,8.006368,10.045521,15.722924,4.48,5.691143,0.003376,0.0,-0.358208,21.401828,-0.272777,10.0
"""75%""","""2026-07-06""",3000.0,8.006368,10.045521,15.72304,4.51,5.731728,0.004144,0.0,1.12977,24.76293,1.321814,14.0
"""max""","""2026-07-11""",3000.0,8.006368,10.045521,15.72304,4.51,5.731728,0.004144,0.0,1.12977,24.76293,1.321814,19.0


In [16]:
FEATURES = [
    "t", "log_m2", "log_fed_bs", "yield_10y", "liq_index",
    "log_s2f", "net_issuance_rate", "activity_composite",
]
TARGET = "log_close"

y = model_df[TARGET].to_numpy()
X = model_df.select(FEATURES).to_numpy()
n_obs = X.shape[0]
X_with_const = np.column_stack([np.ones(n_obs), X])

if n_obs < 60:
    print(f"WARNING: only {n_obs} daily model rows — forecasts will be unstable. "
          "Backfill full ETH history for research-grade intervals.")

_hac_lags = min(30, max(2, n_obs // 5))
ols_model = sm.OLS(y, X_with_const).fit(cov_type="HAC", cov_kwds={"maxlags": _hac_lags})
print(ols_model.summary().tables[1])
print(f"\nR² = {ols_model.rsquared:.4f}  Adj R² = {ols_model.rsquared_adj:.4f}  "
      f"AIC = {ols_model.aic:.1f}  n = {int(ols_model.nobs)}")

best_aic = np.inf
best_order = None
best_arima = None
for p in range(min(4, max(1, n_obs // 20))):
    for q in range(min(4, max(1, n_obs // 20))):
        try:
            m = ARIMA(y, order=(p, 1, q)).fit()
            if m.aic < best_aic:
                best_aic, best_order, best_arima = m.aic, (p, 1, q), m
        except Exception:
            pass
if best_arima is None:
    best_arima = ARIMA(y, order=(1, 1, 0)).fit()
    best_order, best_aic = (1, 1, 0), best_arima.aic
print(f"\nBest ARIMA order: {best_order}  AIC: {best_aic:.1f}")


                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0200   8.03e-18    2.5e+15      0.000       0.020       0.020
x1          5.551e-17   1.55e-16      0.359      0.719   -2.47e-16    3.58e-16
x2             0.2014   8.07e-17    2.5e+15      0.000       0.201       0.201
x3             0.3151   1.24e-16   2.54e+15      0.000       0.315       0.315
x4             0.0894   4.81e-17   1.86e+15      0.000       0.089       0.089
x5            -0.0110   1.14e-15  -9.69e+12      0.000      -0.011      -0.011
x6             0.1087   2.82e-16   3.85e+14      0.000       0.109       0.109
x7          8.777e-05   1.22e-18   7.17e+13      0.000    8.78e-05    8.78e-05
x8            -0.0177   1.42e-15  -1.25e+13      0.000      -0.018      -0.018

R² = -inf  Adj R² = -inf  AIC = -1298.3  n = 20

Best ARIMA order: (0, 1, 0)  AIC: -379.4


/home/ricka/Git/GitHub/ccquant/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


In [17]:
def nearest_psd(cov: np.ndarray) -> np.ndarray:
    cov = np.asarray(cov, dtype=float)
    cov = (cov + cov.T) / 2
    vals, vecs = np.linalg.eigh(cov)
    vals = np.clip(vals, 1e-12, None)
    return (vecs * vals) @ vecs.T

def recent_slope(series: np.ndarray, lookback: int = 730) -> float:
    k = min(lookback, len(series))
    x = np.arange(k)
    return float(np.polyfit(x, series[-k:], 1)[0])

n = model_df.height
Hmax = max(HORIZONS)
future_t = np.arange(n, n + Hmax)
drift_feats = ["log_m2", "log_fed_bs", "yield_10y", "liq_index", "log_s2f",
               "net_issuance_rate", "activity_composite"]
drifts = {f: recent_slope(model_df[f].to_numpy()) for f in drift_feats}

fut = {f: float(model_df[f][-1]) + drifts[f] * np.arange(1, Hmax + 1) for f in drift_feats}
X_future = np.column_stack([
    np.ones(Hmax), future_t,
    fut["log_m2"], fut["log_fed_bs"], fut["yield_10y"], fut["liq_index"],
    fut["log_s2f"], fut["net_issuance_rate"], fut["activity_composite"],
])

cov_psd = nearest_psd(ols_model.cov_params())
coef = ols_model.params
resid = ols_model.resid
rng = np.random.default_rng(RNG_SEED)
paths = np.zeros((BOOTSTRAP_DRAWS, Hmax))
for b in range(BOOTSTRAP_DRAWS):
    beta_draw = rng.multivariate_normal(coef, cov_psd)
    eps = rng.choice(resid, size=Hmax, replace=True)
    paths[b] = np.clip(X_future @ beta_draw + eps, -10, 15)

price_now = float(model_df["close"][-1])
date_now = model_df["date"][-1]
print(f"Current ETH: ${price_now:,.0f}  ({date_now})")

raw_forecasts = {}
for h in HORIZONS:
    p = paths[:, h - 1]
    raw_forecasts[h] = {
        "p50": float(np.exp(np.median(p))),
        "lo50": float(np.exp(np.percentile(p, 25))),
        "hi50": float(np.exp(np.percentile(p, 75))),
        "lo95": float(np.exp(np.percentile(p, 2.5))),
        "hi95": float(np.exp(np.percentile(p, 97.5))),
    }
    f = raw_forecasts[h]
    print(f"  {h:>5}d  median ${f['p50']:>12,.0f}  "
          f"50% [${f['lo50']:>10,.0f}, ${f['hi50']:>10,.0f}]  "
          f"95% [${f['lo95']:>10,.0f}, ${f['hi95']:>10,.0f}]")


Current ETH: $3,000  (2026-07-11)
    365d  median $       3,000  50% [$     2,999, $     3,001]  95% [$     2,998, $     3,002]
    730d  median $       3,000  50% [$     2,998, $     3,001]  95% [$     2,996, $     3,004]
   1461d  median $       3,000  50% [$     2,997, $     3,003]  95% [$     2,992, $     3,008]


In [18]:
def rolling_origin_calibration(horizon: int, step: int = CALIB_STEP_DAYS,
                               draws: int = CALIB_BOOTSTRAP_DRAWS) -> dict:
    scores_50: list[float] = []
    scores_95: list[float] = []
    hit_50 = hit_95 = total = 0
    origins = range(max(min(500, n // 2), n - Hmax - 30, 40), n - horizon, step)
    rng_local = np.random.default_rng(RNG_SEED + horizon)

    for o in origins:
        y_tr = y[:o]
        X_tr = X_with_const[:o]
        m = sm.OLS(y_tr, X_tr).fit()
        cv = nearest_psd(m.cov_params())
        sl = {f: recent_slope(model_df[f].to_numpy()[:o]) for f in drift_feats}
        ft = np.arange(o, o + horizon)
        fut_o = {f: float(model_df[f][o - 1]) + sl[f] * np.arange(1, horizon + 1) for f in drift_feats}
        Xf = np.column_stack([
            np.ones(horizon), ft,
            fut_o["log_m2"], fut_o["log_fed_bs"], fut_o["yield_10y"], fut_o["liq_index"],
            fut_o["log_s2f"], fut_o["net_issuance_rate"], fut_o["activity_composite"],
        ])
        pp = np.zeros(draws)
        for b in range(draws):
            bd = rng_local.multivariate_normal(m.params, cv)
            eps = rng_local.choice(m.resid, size=horizon, replace=True)
            pp[b] = np.clip((Xf @ bd + eps)[-1], -10, 15)
        actual = y[o + horizon - 1]
        med = np.median(pp)
        lo50, hi50 = np.percentile(pp, [25, 75])
        lo95, hi95 = np.percentile(pp, [2.5, 97.5])
        half50 = max((hi50 - lo50) / 2, 1e-8)
        half95 = max((hi95 - lo95) / 2, 1e-8)
        scores_50.append(abs(actual - med) / half50)
        scores_95.append(abs(actual - med) / half95)
        hit_50 += lo50 <= actual <= hi50
        hit_95 += lo95 <= actual <= hi95
        total += 1

    return {
        "horizon": horizon, "origins": total,
        "emp_50": hit_50 / total if total else 0,
        "emp_95": hit_95 / total if total else 0,
        "scale_50": float(np.percentile(scores_50, 50)) if scores_50 else 1.0,
        "scale_95": float(np.percentile(scores_95, 95)) if scores_95 else 1.0,
    }

print("Rolling-origin calibration:")
print(f"  {'horizon':>8} {'origins':>8} {'emp50%':>8} {'emp95%':>8} {'scale50':>8} {'scale95':>8}")
calib_results = {}
for h in HORIZONS:
    r = rolling_origin_calibration(h)
    calib_results[h] = r
    print(f"  {r['horizon']:>8}d {r['origins']:>8} {r['emp_50']:>7.0%} {r['emp_95']:>7.0%} "
          f"{r['scale_50']:>8.2f} {r['scale_95']:>8.2f}")

calibrated = []
for h in HORIZONS:
    raw = raw_forecasts[h]
    sc = calib_results[h]
    med_log = np.log(raw["p50"])
    half50_log = (np.log(raw["hi50"]) - np.log(raw["lo50"])) / 2
    half95_log = (np.log(raw["hi95"]) - np.log(raw["lo95"])) / 2
    s50, s95 = sc["scale_50"], sc["scale_95"]
    calibrated.append({
        "horizon": f"{h}d ({h // 365}y)",
        "date": date_now + timedelta(days=h),
        "median": raw["p50"],
        "lo50_cal": float(np.exp(med_log - half50_log * s50)),
        "hi50_cal": float(np.exp(med_log + half50_log * s50)),
        "lo95_cal": float(np.exp(med_log - half95_log * s95)),
        "hi95_cal": float(np.exp(med_log + half95_log * s95)),
        "emp_50": sc["emp_50"], "emp_95": sc["emp_95"],
        "scale_50": s50, "scale_95": s95,
    })

print(f"\nCurrent ETH: ${price_now:,.0f}  ({date_now})")
for r in calibrated:
    print(f"  {r['horizon']:>12} ${r['median']:>10,.0f}  "
          f"50% [${r['lo50_cal']:>9,.0f}, ${r['hi50_cal']:>9,.0f}]  "
          f"95% [${r['lo95_cal']:>9,.0f}, ${r['hi95_cal']:>9,.0f}]")

arima_fc = best_arima.get_forecast(steps=Hmax)
arima_mean = arima_fc.predicted_mean


Rolling-origin calibration:
   horizon  origins   emp50%   emp95%  scale50  scale95
       365d        0      0%      0%     1.00     1.00
       730d        0      0%      0%     1.00     1.00
      1461d        0      0%      0%     1.00     1.00

Current ETH: $3,000  (2026-07-11)
     365d (1y) $     3,000  50% [$    2,999, $    3,001]  95% [$    2,998, $    3,002]
     730d (2y) $     3,000  50% [$    2,998, $    3,002]  95% [$    2,996, $    3,004]
    1461d (4y) $     3,000  50% [$    2,997, $    3,003]  95% [$    2,992, $    3,008]


In [19]:
n_plot = min(200, BOOTSTRAP_DRAWS)
plot_paths = paths[:n_plot]
future_dates = [date_now + timedelta(days=i + 1) for i in range(Hmax)]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=model_df["date"].to_list(), y=model_df["close"].to_list(),
    mode="lines", name="historical", line=dict(color=ETH_COLOR, width=2),
))
band_edges = [(0, 100), (2.5, 97.5), (5, 95), (10, 90), (25, 75), (40, 60)]
colors = ["rgba(100,100,120,0.08)", "rgba(100,100,120,0.12)", "rgba(100,100,120,0.18)",
          "rgba(100,100,120,0.25)", "rgba(100,100,120,0.35)", "rgba(100,100,120,0.5)"]
for (lo, hi), color in zip(band_edges, colors, strict=True):
    lo_vals = np.exp(np.percentile(plot_paths, lo, axis=0))
    hi_vals = np.exp(np.percentile(plot_paths, hi, axis=0))
    fig.add_trace(go.Scatter(
        x=future_dates + future_dates[::-1],
        y=list(hi_vals) + list(lo_vals[::-1]),
        fill="toself", fillcolor=color, line=dict(color="rgba(0,0,0,0)"),
        name=f"{hi - lo:.0f}% band", hoverinfo="skip",
    ))
fig.add_trace(go.Scatter(
    x=future_dates, y=np.exp(arima_mean), mode="lines", name="ARIMA median",
    line=dict(color="cyan", width=1.5, dash="dash"),
))
for h in HORIZONS:
    r = next(c for c in calibrated if c["horizon"].startswith(f"{h}d"))
    d = date_now + timedelta(days=h)
    fig.add_trace(go.Scatter(
        x=[d], y=[r["median"]], mode="markers+text",
        text=[f"${r['median']:,.0f}"], textposition="top center",
        marker=dict(size=10, color="white"),
        name=f"{h // 365}y forecast", showlegend=(h == HORIZONS[0]),
    ))
fig.update_layout(
    yaxis_type="log",
    title="ETH Long-Term Forecast — OLS Bootstrap (calibrated) + ARIMA",
    template="plotly_dark", height=600, hovermode="x unified",
    legend=dict(orientation="h", y=1.08),
    xaxis=dict(rangeslider=dict(visible=True, thickness=0.05), type="date"),
)
fig.show()


## 9. Summary & caveats

**What this notebook produces**
- Global liquidity regime + early-warning momentum, retargeted to ETH
- ETH on-chain activity composite (fees, TVL, addresses, staking) with live/synthetic sources
- Exploratory **net-issuance S2F** and **fee-based DCF** fair-value band
- Lead/lag evidence, regime backtests, and Probit $P(\text{favorable regime})$
- Calibrated **1y / 2y / 4y** OLS bootstrap fans with ARIMA cross-check

**Caveats**
- OLS is cointegrating, not causal; shared trends inflate $R^2$.
- Synthetic on-chain / FRED fallbacks make numbers **illustrative only** — set API keys for research.
- DCF uses L1 fee revenue as a crude CF proxy; ignores L2 economics, MEV, restaking, etc.
- S2F for ETH is regime-dependent (can be undefined when net-deflationary); capped deliberately.
- In-sample z-scores; live deployment should use expanding-window normalization.
- Research code, **not investment advice**.


In [20]:
latest = df.tail(1)

def _f(col: str, default: float = float("nan")) -> float:
    v = latest[col][0]
    if v is None:
        return default
    try:
        x = float(v)
        return x if x == x else default
    except Exception:
        return default

print("=" * 72)
print("ETH VALUE / SIGNAL SNAPSHOT")
print("=" * 72)
print(f"Latest week:              {latest['date'][0]}")
print(f"ETH price:                ${_f('close'):,.0f}")
print(f"Liquidity regime:         {'EXPANDING' if int(_f('regime', 0)) else 'CONTRACTING'}")
print(f"Macro early-warn:         {_f('early_warn'):+.2f}")
print(f"ETH early composite:      {_f('eth_early'):+.2f}")
print(f"Activity composite:       {_f('activity_composite'):+.2f}")
print(f"S2F:                      {_f('s2f'):.1f}  "
      f"(deflationary={bool(int(_f('is_deflationary', 0)))})")
print(f"Net issuance rate:        {_f('net_issuance_rate') * 100:.2f}% / yr")
_dcf = _f("dcf_price_base")
_gap = _f("dcf_gap")
if _dcf == _dcf:
    print(f"DCF base fair value:      ${_dcf:,.0f}  (gap {_gap:+.1%})")
else:
    print("DCF base fair value:      n/a")
if p_fav == p_fav:  # not NaN
    print(f"P(favorable in 13w):      {p_fav:.0%}")
else:
    print("P(favorable in 13w):      n/a (short history)")
print()
print("Calibrated forecast medians:")
for r in calibrated:
    print(f"  {r['horizon']:>12}: ${r['median']:>10,.0f}")
print()
_brier = f"{float(np.mean(oos_brier)):.3f}" if oos_brier else "n/a"
print(f"OLS R² = {ols_model.rsquared:.4f}  ARIMA order = {best_order}  "
      f"Probit Brier = {_brier}")
print(f"On-chain source tag: {oc_source}")


ETH VALUE / SIGNAL SNAPSHOT
Latest week:              2026-07-06
ETH price:                $3,000
Liquidity regime:         CONTRACTING
Macro early-warn:         +nan
ETH early composite:      +nan
Activity composite:       -0.36
S2F:                      241.3  (deflationary=False)
Net issuance rate:        0.41% / yr
DCF base fair value:      $116  (gap +2476.3%)
P(favorable in 13w):      n/a (short history)

Calibrated forecast medians:
     365d (1y): $     3,000
     730d (2y): $     3,000
    1461d (4y): $     3,000

OLS R² = -inf  ARIMA order = (0, 1, 0)  Probit Brier = n/a
On-chain source tag: synthetic+defillama
